**Data Extraction**

In [1]:
import pandas as pd
import requests as re
import numpy as np

In [2]:
df = pd.read_csv("IPL_Dataset(2008-2024).csv")

In [3]:
df.head()

,Match_ID,Date,Teams,Venue,Toss_Winner,Toss_Decision,Match_Winner,Win_Type,Win_Margin,First_Innings_Score,Second_Innings_Score,Player_of_Match,Umpire,Umpire1,Umpire2,Powerplay_Scores,Middle_Overs_Scores,Death_Overs_Scores
0,335982,2008-04-18,Royal Challengers Bangalore vs Kolkata Knight ...,M Chinnaswamy Stadium,Royal Challengers Bangalore,field,Kolkata Knight Riders,runs,140.0,222,82.0,BB McCullum,Asad Rauf,RE Koertzen,J Srinath,61,97,64
1,335983,2008-04-19,Kings XI Punjab vs Chennai Super Kings,"Punjab Cricket Association Stadium, Mohali",Chennai Super Kings,bat,Chennai Super Kings,runs,33.0,240,207.0,MEK Hussey,MR Benson,SL Shastri,S Venkataraghavan,53,116,71
2,335984,2008-04-19,Delhi Daredevils vs Rajasthan Royals,Feroz Shah Kotla,Rajasthan Royals,bat,Delhi Daredevils,wickets,9.0,129,132.0,MF Maharoof,Aleem Dar,GA Pratapkumar,GR Viswanath,40,66,23
3,335985,2008-04-20,Mumbai Indians vs Royal Challengers Bangalore,Wankhede Stadium,Mumbai Indians,bat,Royal Challengers Bangalore,wickets,5.0,165,166.0,MV Boucher,SJ Davis,DJ Harper,J Srinath,47,71,47
4,335986,2008-04-20,Kolkata Knight Riders vs Deccan Chargers,Eden Gardens,Deccan Chargers,bat,Kolkata Knight Riders,wickets,5.0,110,112.0,DJ Hussey,BF Bowden,K Hariharan,FM Engineer,39,43,28


In [4]:
df.describe()

,Match_ID,Win_Margin,First_Innings_Score,Second_Innings_Score,Powerplay_Scores,Middle_Overs_Scores,Death_Overs_Scores
count,1.073000e+03,1054.000000,1073.000000,1070.000000,1073.000000,1073.000000,1073.000000
mean,8.941365e+05,17.158444,165.164958,151.918692,46.074557,78.248835,40.841566
std,3.637486e+05,21.762303,31.904238,31.751666,12.007306,18.604148,14.228121
min,3.359820e+05,1.000000,56.000000,2.000000,15.000000,0.000000,0.000000
25%,5.483260e+05,6.000000,145.000000,134.000000,38.000000,66.000000,33.000000
50%,9.809390e+05,8.000000,165.000000,153.000000,46.000000,77.000000,41.000000
75%,1.216540e+06,19.750000,186.000000,172.000000,53.000000,90.000000,50.000000
max,1.426287e+06,146.000000,287.000000,262.000000,125.000000,155.000000,89.000000


In [5]:
df.shape

(1073, 18)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1073 entries, 0 to 1072
Data columns (total 18 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Match_ID              1073 non-null   int64  
 1   Date                  1073 non-null   object 
 2   Teams                 1073 non-null   object 
 3   Venue                 1073 non-null   object 
 4   Toss_Winner           1073 non-null   object 
 5   Toss_Decision         1073 non-null   object 
 6   Match_Winner          1073 non-null   object 
 7   Win_Type              1054 non-null   object 
 8   Win_Margin            1054 non-null   float64
 9   First_Innings_Score   1073 non-null   int64  
 10  Second_Innings_Score  1070 non-null   float64
 11  Player_of_Match       1068 non-null   object 
 12  Umpire                1073 non-null   object 
 13  Umpire1               1073 non-null   object 
 14  Umpire2               1073 non-null   object 
 15  Powerplay_Scores     

In [7]:
df = pd.read_csv("IPL_Dataset(2008-2024).csv")
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date")

# Team1 / Team2
teams = df["Teams"].str.split(" vs ", n=1, expand=True)
df["Team1"] = teams[0].str.strip()
df["Team2"] = teams[1].str.strip()

# Long format (two rows per match)
long = pd.concat([
    df[["Match_ID","Date","Team1","Team2","Match_Winner"]].rename(columns={"Team1":"Team","Team2":"Opp"}),
    df[["Match_ID","Date","Team2","Team1","Match_Winner"]].rename(columns={"Team2":"Team","Team1":"Opp"})
], ignore_index=True)

long["win"] = (long["Match_Winner"] == long["Team"]).astype(int)

# Recent form (previous N matches)
N = 5
long[f"form_winrate_{N}"] = (
    long.groupby("Team")["win"]
        .transform(lambda s: s.shift(1).rolling(N, min_periods=1).mean())
)

# Merge back: Team1 form + Team2 form
t1 = long.merge(df[["Match_ID","Team1"]], on="Match_ID")
t1 = t1[t1["Team"] == t1["Team1"]][["Match_ID", f"form_winrate_{N}"]].rename(
    columns={f"form_winrate_{N}": f"team1_form_winrate_{N}"}
)

t2 = long.merge(df[["Match_ID","Team2"]], on="Match_ID")
t2 = t2[t2["Team"] == t2["Team2"]][["Match_ID", f"form_winrate_{N}"]].rename(
    columns={f"form_winrate_{N}": f"team2_form_winrate_{N}"}
)

df2 = df.merge(t1, on="Match_ID").merge(t2, on="Match_ID")


In [8]:
df2

,Match_ID,Date,Teams,Venue,Toss_Winner,Toss_Decision,Match_Winner,Win_Type,Win_Margin,First_Innings_Score,...,Umpire,Umpire1,Umpire2,Powerplay_Scores,Middle_Overs_Scores,Death_Overs_Scores,Team1,Team2,team1_form_winrate_5,team2_form_winrate_5
0,335982,2008-04-18,Royal Challengers Bangalore vs Kolkata Knight ...,M Chinnaswamy Stadium,Royal Challengers Bangalore,field,Kolkata Knight Riders,runs,140.0,222,...,Asad Rauf,RE Koertzen,J Srinath,61,97,64,Royal Challengers Bangalore,Kolkata Knight Riders,NaN,0.4
1,335983,2008-04-19,Kings XI Punjab vs Chennai Super Kings,"Punjab Cricket Association Stadium, Mohali",Chennai Super Kings,bat,Chennai Super Kings,runs,33.0,240,...,MR Benson,SL Shastri,S Venkataraghavan,53,116,71,Kings XI Punjab,Chennai Super Kings,NaN,0.4
2,335984,2008-04-19,Delhi Daredevils vs Rajasthan Royals,Feroz Shah Kotla,Rajasthan Royals,bat,Delhi Daredevils,wickets,9.0,129,...,Aleem Dar,GA Pratapkumar,GR Viswanath,40,66,23,Delhi Daredevils,Rajasthan Royals,NaN,0.4
3,335985,2008-04-20,Mumbai Indians vs Royal Challengers Bangalore,Wankhede Stadium,Mumbai Indians,bat,Royal Challengers Bangalore,wickets,5.0,165,...,SJ Davis,DJ Harper,J Srinath,47,71,47,Mumbai Indians,Royal Challengers Bangalore,NaN,0.4
4,335986,2008-04-20,Kolkata Knight Riders vs Deccan Chargers,Eden Gardens,Deccan Chargers,bat,Kolkata Knight Riders,wickets,5.0,110,...,BF Bowden,K Hariharan,FM Engineer,39,43,28,Kolkata Knight Riders,Deccan Chargers,NaN,0.6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1068,1426284,2024-04-28,Chennai Super Kings vs Sunrisers Hyderabad,"MA Chidambaram Stadium, Chepauk, Chennai",Sunrisers Hyderabad,field,Chennai Super Kings,runs,78.0,212,...,R Pandit,MV Saidharshan Kumar,Sanjay Verma,50,109,53,Chennai Super Kings,Sunrisers Hyderabad,0.4,0.4
1069,1426283,2024-04-28,Gujarat Titans vs Royal Challengers Bengaluru,"Narendra Modi Stadium, Ahmedabad",Royal Challengers Bengaluru,field,Royal Challengers Bengaluru,wickets,9.0,200,...,Nitin Menon,VK Sharma,J Srinath,42,106,52,Gujarat Titans,Royal Challengers Bengaluru,0.4,0.4
1070,1426285,2024-04-29,Delhi Capitals vs Kolkata Knight Riders,"Eden Gardens, Kolkata",Delhi Capitals,bat,Kolkata Knight Riders,wickets,7.0,153,...,Navdeep Singh,Tapan Sharma,V Narayan Kutty,67,61,25,Delhi Capitals,Kolkata Knight Riders,0.8,0.8
1071,1426286,2024-04-30,Mumbai Indians vs Lucknow Super Giants,Bharat Ratna Shri Atal Bihari Vajpayee Ekana C...,Lucknow Super Giants,field,Lucknow Super Giants,wickets,4.0,144,...,MA Gough,UV Gandhe,DS Manohar,28,78,38,Mumbai Indians,Lucknow Super Giants,0.6,0.6


In [9]:
# Split teams
t = df2["Teams"].str.split(" vs ", n=1, expand=True)
df2["Team1"] = t[0].str.strip()
df2["Team2"] = t[1].str.strip()
df2["Toss_Decision"] = df2["Toss_Decision"].str.lower().str.strip()

def other(team, team1, team2):
    return team2 if team == team1 else team1

def bat_first(row):
    if row["Toss_Decision"] == "bat":
        return row["Toss_Winner"]
    if row["Toss_Decision"] == "field":
        return other(row["Toss_Winner"], row["Team1"], row["Team2"])
    return np.nan

df2["Bat_First"] = df2.apply(bat_first, axis=1)
df2["Bat_Second"] = df2.apply(lambda r: other(r["Bat_First"], r["Team1"], r["Team2"])
                            if isinstance(r["Bat_First"], str) else np.nan, axis=1)

venue_games = {}
venue_chase_wins = {}

alpha = 1  # smoothing (Laplace)

venue_score_prior = []
venue_chase_rate_prior = []

for _, r in df2.iterrows():
    v = r["Venue"]
    g = venue_games.get(v, 0)
    cw = venue_chase_wins.get(v, 0)

    # prior chase win rate (smoothed)
    chase_rate = (cw + alpha) / (g + 2*alpha) if g > 0 else 0.5
    score = 2 * chase_rate - 1

    venue_chase_rate_prior.append(chase_rate)
    venue_score_prior.append(score)

    # update with current match result
    if isinstance(r["Bat_Second"], str):
        venue_games[v] = g + 1
        if r["Match_Winner"] == r["Bat_Second"]:
            venue_chase_wins[v] = cw + 1

df2["venue_chase_winrate_prior"] = venue_chase_rate_prior
df2["venue_score_prior"] = venue_score_prior


In [10]:
# 1) Rename teams everywhere
mapping = {
    "Delhi Daredevils": "Delhi Capitals",
    "Kings XI Punjab": "Punjab Kings",
    "Royal Challengers Bangalore": "Royal Challengers Bengaluru",
}

# Replace inside "Teams" text and in single-team columns
for old, new in mapping.items():
    df2["Teams"] = df2["Teams"].str.replace(old, new, regex=False)

for c in ["Toss_Winner", "Match_Winner"]:
    df2[c] = df2[c].replace(mapping)

# Re-split Team1/Team2 after renaming (useful for filtering)
t = df2["Teams"].str.split(" vs ", n=1, expand=True)
df2["Team1"] = t[0].str.strip()
df2["Team2"] = t[1].str.strip()

# 2) Drop teams that are not playing anymore
defunct = {
    "Deccan Chargers",
    "Gujarat Lions",
    "Kochi Tuskers Kerala",
    "Pune Warriors",
    "Rising Pune Supergiants",
}

df2 = df2[~(df2["Team1"].isin(defunct) | df2["Team2"].isin(defunct))].copy()


In [11]:
df2

,Match_ID,Date,Teams,Venue,Toss_Winner,Toss_Decision,Match_Winner,Win_Type,Win_Margin,First_Innings_Score,...,Middle_Overs_Scores,Death_Overs_Scores,Team1,Team2,team1_form_winrate_5,team2_form_winrate_5,Bat_First,Bat_Second,venue_chase_winrate_prior,venue_score_prior
0,335982,2008-04-18,Royal Challengers Bengaluru vs Kolkata Knight ...,M Chinnaswamy Stadium,Royal Challengers Bengaluru,field,Kolkata Knight Riders,runs,140.0,222,...,97,64,Royal Challengers Bengaluru,Kolkata Knight Riders,NaN,0.4,Kolkata Knight Riders,Royal Challengers Bangalore,0.500000,0.000000
1,335983,2008-04-19,Punjab Kings vs Chennai Super Kings,"Punjab Cricket Association Stadium, Mohali",Chennai Super Kings,bat,Chennai Super Kings,runs,33.0,240,...,116,71,Punjab Kings,Chennai Super Kings,NaN,0.4,Chennai Super Kings,Kings XI Punjab,0.500000,0.000000
2,335984,2008-04-19,Delhi Capitals vs Rajasthan Royals,Feroz Shah Kotla,Rajasthan Royals,bat,Delhi Capitals,wickets,9.0,129,...,66,23,Delhi Capitals,Rajasthan Royals,NaN,0.4,Rajasthan Royals,Delhi Daredevils,0.500000,0.000000
3,335985,2008-04-20,Mumbai Indians vs Royal Challengers Bengaluru,Wankhede Stadium,Mumbai Indians,bat,Royal Challengers Bengaluru,wickets,5.0,165,...,71,47,Mumbai Indians,Royal Challengers Bengaluru,NaN,0.4,Mumbai Indians,Royal Challengers Bangalore,0.500000,0.000000
5,335987,2008-04-21,Rajasthan Royals vs Punjab Kings,Sawai Mansingh Stadium,Punjab Kings,bat,Rajasthan Royals,wickets,6.0,166,...,74,38,Rajasthan Royals,Punjab Kings,NaN,0.2,Kings XI Punjab,Rajasthan Royals,0.500000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1068,1426284,2024-04-28,Chennai Super Kings vs Sunrisers Hyderabad,"MA Chidambaram Stadium, Chepauk, Chennai",Sunrisers Hyderabad,field,Chennai Super Kings,runs,78.0,212,...,109,53,Chennai Super Kings,Sunrisers Hyderabad,0.4,0.4,Chennai Super Kings,Sunrisers Hyderabad,0.480000,-0.040000
1069,1426283,2024-04-28,Gujarat Titans vs Royal Challengers Bengaluru,"Narendra Modi Stadium, Ahmedabad",Royal Challengers Bengaluru,field,Royal Challengers Bengaluru,wickets,9.0,200,...,106,52,Gujarat Titans,Royal Challengers Bengaluru,0.4,0.4,Gujarat Titans,Royal Challengers Bengaluru,0.590909,0.181818
1070,1426285,2024-04-29,Delhi Capitals vs Kolkata Knight Riders,"Eden Gardens, Kolkata",Delhi Capitals,bat,Kolkata Knight Riders,wickets,7.0,153,...,61,25,Delhi Capitals,Kolkata Knight Riders,0.8,0.8,Delhi Capitals,Kolkata Knight Riders,0.500000,0.000000
1071,1426286,2024-04-30,Mumbai Indians vs Lucknow Super Giants,Bharat Ratna Shri Atal Bihari Vajpayee Ekana C...,Lucknow Super Giants,field,Lucknow Super Giants,wickets,4.0,144,...,78,38,Mumbai Indians,Lucknow Super Giants,0.6,0.6,Mumbai Indians,Lucknow Super Giants,0.428571,-0.142857


In [12]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
Index: 902 entries, 0 to 1072
Data columns (total 26 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   Match_ID                   902 non-null    int64         
 1   Date                       902 non-null    datetime64[ns]
 2   Teams                      902 non-null    object        
 3   Venue                      902 non-null    object        
 4   Toss_Winner                902 non-null    object        
 5   Toss_Decision              902 non-null    object        
 6   Match_Winner               902 non-null    object        
 7   Win_Type                   885 non-null    object        
 8   Win_Margin                 885 non-null    float64       
 9   First_Innings_Score        902 non-null    int64         
 10  Second_Innings_Score       900 non-null    float64       
 11  Player_of_Match            898 non-null    object        
 12  Umpire      

In [13]:
new_df = df2[["Match_ID","Date","Teams","Team1","Team2","Toss_Winner","Toss_Decision","team1_form_winrate_5","team2_form_winrate_5","venue_chase_winrate_prior","venue_score_prior","Match_Winner"]].reset_index(drop=True)

In [14]:
new_df

,Match_ID,Date,Teams,Team1,Team2,Toss_Winner,Toss_Decision,team1_form_winrate_5,team2_form_winrate_5,venue_chase_winrate_prior,venue_score_prior,Match_Winner
0,335982,2008-04-18,Royal Challengers Bengaluru vs Kolkata Knight ...,Royal Challengers Bengaluru,Kolkata Knight Riders,Royal Challengers Bengaluru,field,NaN,0.4,0.500000,0.000000,Kolkata Knight Riders
1,335983,2008-04-19,Punjab Kings vs Chennai Super Kings,Punjab Kings,Chennai Super Kings,Chennai Super Kings,bat,NaN,0.4,0.500000,0.000000,Chennai Super Kings
2,335984,2008-04-19,Delhi Capitals vs Rajasthan Royals,Delhi Capitals,Rajasthan Royals,Rajasthan Royals,bat,NaN,0.4,0.500000,0.000000,Delhi Capitals
3,335985,2008-04-20,Mumbai Indians vs Royal Challengers Bengaluru,Mumbai Indians,Royal Challengers Bengaluru,Mumbai Indians,bat,NaN,0.4,0.500000,0.000000,Royal Challengers Bengaluru
4,335987,2008-04-21,Rajasthan Royals vs Punjab Kings,Rajasthan Royals,Punjab Kings,Punjab Kings,bat,NaN,0.2,0.500000,0.000000,Rajasthan Royals
...,...,...,...,...,...,...,...,...,...,...,...,...
897,1426284,2024-04-28,Chennai Super Kings vs Sunrisers Hyderabad,Chennai Super Kings,Sunrisers Hyderabad,Sunrisers Hyderabad,field,0.4,0.4,0.480000,-0.040000,Chennai Super Kings
898,1426283,2024-04-28,Gujarat Titans vs Royal Challengers Bengaluru,Gujarat Titans,Royal Challengers Bengaluru,Royal Challengers Bengaluru,field,0.4,0.4,0.590909,0.181818,Royal Challengers Bengaluru
899,1426285,2024-04-29,Delhi Capitals vs Kolkata Knight Riders,Delhi Capitals,Kolkata Knight Riders,Delhi Capitals,bat,0.8,0.8,0.500000,0.000000,Kolkata Knight Riders
900,1426286,2024-04-30,Mumbai Indians vs Lucknow Super Giants,Mumbai Indians,Lucknow Super Giants,Lucknow Super Giants,field,0.6,0.6,0.428571,-0.142857,Lucknow Super Giants


In [15]:
new_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 902 entries, 0 to 901
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   Match_ID                   902 non-null    int64         
 1   Date                       902 non-null    datetime64[ns]
 2   Teams                      902 non-null    object        
 3   Team1                      902 non-null    object        
 4   Team2                      902 non-null    object        
 5   Toss_Winner                902 non-null    object        
 6   Toss_Decision              902 non-null    object        
 7   team1_form_winrate_5       890 non-null    float64       
 8   team2_form_winrate_5       902 non-null    float64       
 9   venue_chase_winrate_prior  902 non-null    float64       
 10  venue_score_prior          902 non-null    float64       
 11  Match_Winner               902 non-null    object        
dtypes: datet

In [16]:
output_path = "IPL_Winner_Model_Dataset.csv"
new_df.to_csv(output_path, index=False)
print(f"Saved: {output_path}")

Saved: IPL_Winner_Model_Dataset.csv
